# ANEXO — EDA Clásico (resumido) sobre Capa Semántica (SEM)
**ChileCompra Data Platform — TFM (UCM)**

> **Propósito**: Este notebook presenta un **EDA clásico resumido** (estadística descriptiva + visualizaciones) ejecutado **exclusivamente** sobre la capa semántica del Data Warehouse (**`dw_sem`**), que corresponde al **dataset gobernado consumido por Power BI**.  
> El EDA principal del proyecto se realizó sobre **RAW** (Data Governance / Data Quality Profiling / Matching Diagnosis). Este anexo **complementa** dicho enfoque sin sustituirlo.

**Restricciones metodológicas**  
- No usar RAW directamente.  
- No imputar, no corregir, no modificar tablas. Diagnóstico únicamente.  
- **Segmentación obligatoria por `moneda_norm`** (prohibido sumar CLP+USD+CLF/EUR/UTM).  
- Guardar outputs reproducibles en: `./outputs/ANEXO_EDA_SEM/` (CSV y PNG).

**Vistas oficiales (contrato analítico)**  
- `dw_sem.v_fact_licitaciones_sem_v2`  
- `dw_sem.v_fact_ordenes_compra_sem_v2`


## 0 — Setup (dependencias, rutas, conexión)

In [1]:
# =========================================
# 0.1 - Imports
# =========================================
import os
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

from sqlalchemy import create_engine, text

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)


In [2]:
# =========================================
# 0.2 - load_config() (sin secretos hardcodeados)
# =========================================
def load_config():
    """
    Carga configuración desde variables de entorno.
    - No escribe ni modifica archivos.
    - Compatible con ejecución local o contenedor.
    """
    cfg = {
        # Preferencia: variables del repo
        "PG_HOST": os.getenv("NOTEBOOK_PG_HOST", os.getenv("CHILE_PG_HOST", "localhost")),
        "PG_PORT": int(os.getenv("NOTEBOOK_PG_PORT", os.getenv("POSTGRES_PORT", "5433"))),
        "PG_DB":   os.getenv("CHILE_PG_DB", os.getenv("POSTGRES_DB", "chilecompra")),
        "PG_USER": os.getenv("CHILE_PG_USER", os.getenv("POSTGRES_USER", "chile_user")),
        "PG_PASS": os.getenv("CHILE_PG_PASSWORD", os.getenv("POSTGRES_PASSWORD", "")),
        "OUTPUT_DIR": os.getenv("OUTPUT_DIR", "./outputs/ANEXO_EDA_SEM"),
    }
    return cfg

cfg = load_config()
cfg


{'PG_HOST': 'localhost',
 'PG_PORT': 5433,
 'PG_DB': 'chilecompra',
 'PG_USER': 'chile_user',
 'PG_PASS': 'ChilePg_2024!',
 'OUTPUT_DIR': './outputs/ANEXO_EDA_SEM'}

In [3]:
# =========================================
# 0.3 - Output helpers
# =========================================
OUTDIR = Path(cfg["OUTPUT_DIR"]).resolve()
OUTDIR.mkdir(parents=True, exist_ok=True)

def safe_to_csv(df: pd.DataFrame, filename: str):
    p = OUTDIR / filename
    df.to_csv(p, index=False, encoding="utf-8")
    return p

def ensure_fig_saved(filename: str, title: str = None):
    if title:
        plt.title(title)
    p = OUTDIR / filename
    plt.tight_layout()
    plt.savefig(p, dpi=150)
    plt.close()
    return p

OUTDIR


PosixPath('/home/engineer/Documents/Proyectos/TFM/docs/evidence/Fase6_EDA_CLASICO/outputs/ANEXO_EDA_SEM')

In [4]:
# =========================================
# 0.4 - Conexión Postgres (SQLAlchemy)
# =========================================
# Nota:
# - En ejecución LOCAL, normalmente PG_HOST=localhost y PG_PORT=5433 (puerto publicado).
# - En ejecución dentro de Docker network, PG_HOST puede ser chile-pg y puerto 5432.
# Ajusta NOTEBOOK_PG_HOST/NOTEBOOK_PG_PORT o variables del repo según corresponda.

dsn = f"postgresql+psycopg2://{cfg['PG_USER']}:{cfg['PG_PASS']}@{cfg['PG_HOST']}:{cfg['PG_PORT']}/{cfg['PG_DB']}"
engine = create_engine(dsn, pool_pre_ping=True)

def read_sql_df(q: str, params=None) -> pd.DataFrame:
    with engine.connect() as conn:
        return pd.read_sql(text(q), conn, params=params or {})

# Smoke test (opcional)
read_sql_df("SELECT 1 AS ok;")


,ok
0,1


## 1 — Contexto y justificación metodológica (UCM)

**Texto listo para pegar en el informe**  
El EDA principal del proyecto ChileCompra Data Platform se desarrolló sobre la capa RAW con enfoque de Data Governance, Data Quality Profiling y Matching Diagnosis LIC–OC, dado que dicho nivel permite auditar continuidad mensual, estructura de archivos, duplicidad y anomalías de fuente. En complemento, este anexo presenta un EDA clásico resumido ejecutado sobre la capa semántica del Data Warehouse (`dw_sem`), ya que corresponde al dataset gobernado que consume Power BI. Por tanto, el análisis se centra en describir distribución, completitud y comportamiento temporal del universo analítico final, sin sustituir la evidencia de gobernanza ni aplicar procesos de limpieza, imputación o modificación de datos.


## 2 — Magnitud global del dataset SEM (LIC y OC)

**Objetivo**: dimensionar el universo analítico (volumen, cobertura, diversidad) y dejar evidencia exportable.


In [5]:
# =========================================
# 2.1 - Detección de columnas clave (robusto)
# =========================================
def get_columns(schema: str, view: str) -> list[str]:
    q = """
    SELECT column_name
    FROM information_schema.columns
    WHERE table_schema=:schema AND table_name=:view
    ORDER BY ordinal_position;
    """
    return read_sql_df(q, {"schema": schema, "view": view})["column_name"].tolist()

lic_cols = get_columns("dw_sem", "v_fact_licitaciones_sem_v2")
oc_cols  = get_columns("dw_sem", "v_fact_ordenes_compra_sem_v2")

lic_cols[:20], oc_cols[:20]


(['licitacion_sk',
  'licitacion_bk',
  'fecha_publicacion_sk',
  'fecha_cierre_sk',
  'organismo_sk',
  'proveedor_sk',
  'producto_onu_sk',
  'monto_estimado',
  'monto_adjudicado',
  'cantidad_oferentes',
  'cantidad_items',
  'estado_licitacion',
  'tipo_adquisicion',
  'moneda',
  'periodo',
  'source_file',
  'ingested_at',
  'moneda_raw',
  'moneda_norm',
  'flag_moneda_missing'],
 ['orden_compra_sk',
  'orden_compra_bk',
  'fecha_creacion_sk',
  'fecha_aceptacion_sk',
  'organismo_sk',
  'proveedor_sk',
  'producto_onu_sk',
  'monto_total',
  'cantidad_total',
  'cantidad_items',
  'estado_oc',
  'tipo_oc',
  'moneda',
  'periodo',
  'source_file',
  'ingested_at',
  'moneda_raw',
  'moneda_norm',
  'flag_moneda_missing',
  'flag_moneda_unknown'])

In [6]:
# =========================================
# 2.2 - Magnitud global LIC / OC (total + por moneda + diversidad)
# =========================================
def pick_first(cols, cands):
    s = {c.lower(): c for c in cols}
    for x in cands:
        if x.lower() in s:
            return s[x.lower()]
    return None

lic_mon = pick_first(lic_cols, ["moneda_norm", "moneda"])
oc_mon  = pick_first(oc_cols,  ["moneda_norm", "moneda"])

lic_org_sk = pick_first(lic_cols, ["organismo_sk"])
oc_org_sk  = pick_first(oc_cols,  ["organismo_sk"])

lic_prov_sk = pick_first(lic_cols, ["proveedor_sk"])
oc_prov_sk  = pick_first(oc_cols,  ["proveedor_sk"])

lic_prod_sk = pick_first(lic_cols, ["producto_onu_sk", "producto_sk"])
oc_prod_sk  = pick_first(oc_cols,  ["producto_onu_sk", "producto_sk"])

def magnitud_fact(view: str, moneda_col: str, org_col: str, prov_col: str, prod_col: str):
    moneda_expr = moneda_col if moneda_col else "NULL::text"
    org_expr    = org_col if org_col else "NULL::bigint"
    prov_expr   = prov_col if prov_col else "NULL::bigint"
    prod_expr   = prod_col if prod_col else "NULL::bigint"
    q = f"""
    SELECT
        {moneda_expr}::text AS moneda_norm,
        COUNT(*)::bigint AS n_registros,
        COUNT(DISTINCT {org_expr})::bigint  AS n_organismos_dist,
        COUNT(DISTINCT {prov_expr})::bigint AS n_proveedores_dist,
        COUNT(DISTINCT {prod_expr})::bigint AS n_productos_onu_dist
    FROM dw_sem.{view}
    GROUP BY 1
    ORDER BY n_registros DESC;
    """
    return read_sql_df(q)

df_mag_lic = magnitud_fact("v_fact_licitaciones_sem_v2", lic_mon, lic_org_sk, lic_prov_sk, lic_prod_sk)
df_mag_oc  = magnitud_fact("v_fact_ordenes_compra_sem_v2", oc_mon,  oc_org_sk,  oc_prov_sk,  oc_prod_sk)

safe_to_csv(df_mag_lic, "tabla_magnitud_sem_lic.csv")
safe_to_csv(df_mag_oc,  "tabla_magnitud_sem_oc.csv")

df_mag_lic, df_mag_oc


(  moneda_norm  n_registros  n_organismos_dist  n_proveedores_dist  n_productos_onu_dist
 0         CLP      4620401                846               19425                  8304
 1         USD         7329                110                 567                   250
 2         CLF         4761                237                1033                   322
 3        None          590                 24                 197                    76,
   moneda_norm  n_registros  n_organismos_dist  n_proveedores_dist  n_productos_onu_dist
 0         CLP      4531314                845               37550                  9716
 1         USD        12744                607                 620                   525
 2         CLF        11901                573                1114                   688
 3         UTM          818                 50                 129                   130
 4         EUR           35                 15                  18                    20)

In [7]:
# =========================================
# 2.3 - Gráfico: barras agrupadas LIC vs OC por moneda_norm
# =========================================
df_plot = (
    df_mag_lic[["moneda_norm","n_registros"]].rename(columns={"n_registros":"n_lic"})
    .merge(df_mag_oc[["moneda_norm","n_registros"]].rename(columns={"n_registros":"n_oc"}),
           on="moneda_norm", how="outer")
    .fillna(0)
)

df_plot["n_total"] = df_plot["n_lic"] + df_plot["n_oc"]
df_plot = df_plot.sort_values("n_total", ascending=False)

x = np.arange(len(df_plot))
w = 0.4

plt.figure(figsize=(10,4))
plt.bar(x - w/2, df_plot["n_lic"], width=w, label="LIC")
plt.bar(x + w/2, df_plot["n_oc"],  width=w, label="OC")
plt.xticks(x, df_plot["moneda_norm"].astype(str), rotation=0)
plt.ylabel("Nº registros")
plt.legend()
ensure_fig_saved("fig_barras_lic_oc_por_moneda.png", "Fig. {A}.1 — Volumen LIC vs OC por moneda_norm")


PosixPath('/home/engineer/Documents/Proyectos/TFM/docs/evidence/Fase6_EDA_CLASICO/outputs/ANEXO_EDA_SEM/fig_barras_lic_oc_por_moneda.png')

## 3 — Completitud (missing) y cardinalidad

**Objetivo**: reportar columnas con mayor incompletitud en SEM (Top 25 por dominio), incluyendo % nulos, % no nulos y cardinalidad.


In [8]:
# =========================================
# 3.1 - Muestra controlada (evita scans pesados)
# =========================================
SAMPLE_FRAC = float(os.getenv("EDA_SAMPLE_FRAC", "0.15"))  # 15% por defecto
SAMPLE_SEED = int(os.getenv("EDA_SAMPLE_SEED", "42"))

def sample_df(view: str, frac: float):
    if frac >= 1.0:
        return read_sql_df(f"SELECT * FROM dw_sem.{view};")
    q = f"""
    SELECT *
    FROM dw_sem.{view}
    TABLESAMPLE SYSTEM ({max(1.0, frac*100):.2f})
    REPEATABLE({SAMPLE_SEED});
    """
    try:
        return read_sql_df(q)
    except Exception:
        n = int(200000 * frac)
        return read_sql_df(f"SELECT * FROM dw_sem.{view} ORDER BY random() LIMIT {n};")

df_lic_s = sample_df("v_fact_licitaciones_sem_v2", SAMPLE_FRAC)
df_oc_s  = sample_df("v_fact_ordenes_compra_sem_v2", SAMPLE_FRAC)

df_lic_s.shape, df_oc_s.shape


((30000, 28), (30000, 25))

In [9]:
# =========================================
# 3.2 - Missing + cardinalidad (Top 25)
# =========================================
def missing_profile_top(df: pd.DataFrame, topk: int = 25) -> pd.DataFrame:
    n = len(df)
    rows = []
    for c in df.columns:
        nulls = int(df[c].isna().sum())
        rows.append({
            "columna": c,
            "pct_nulos": round(100*nulls/n, 4) if n else None,
            "pct_no_nulos": round(100*(1 - nulls/n), 4) if n else None,
            "n_distinct": int(df[c].nunique(dropna=True))
        })
    out = pd.DataFrame(rows).sort_values(["pct_nulos","n_distinct"], ascending=[False, True]).head(topk)
    return out

miss_lic = missing_profile_top(df_lic_s, 25)
miss_oc  = missing_profile_top(df_oc_s,  25)

safe_to_csv(miss_lic, "tabla_missing_top25_lic.csv")
safe_to_csv(miss_oc,  "tabla_missing_top25_oc.csv")

miss_lic, miss_oc


(                           columna  pct_nulos  pct_no_nulos  n_distinct
 3                  fecha_cierre_sk    44.8133       55.1867         206
 8                 monto_adjudicado    12.7233       87.2767        7675
 21              sector_estado_norm     0.7067       99.2933           7
 7                   monto_estimado     0.6800       99.3200        4732
 27             monto_estimado_norm     0.6800       99.3200        4732
 18                     moneda_norm     0.0200       99.9800           3
 10                  cantidad_items     0.0000      100.0000           1
 19             flag_moneda_missing     0.0000      100.0000           1
 23             flag_sector_unknown     0.0000      100.0000           1
 24           flag_organismo_valido     0.0000      100.0000           1
 25  flag_fuente_preferente_missing     0.0000      100.0000           1
 26        flag_monto_estimado_anom     0.0000      100.0000           1
 20             flag_moneda_unknown     0.0000     

In [10]:
# =========================================
# 3.3 - Gráficos: barras horizontales Top 25 nulos
# =========================================
def plot_missing(df_miss: pd.DataFrame, filename: str, title: str):
    plt.figure(figsize=(10,6))
    plt.barh(df_miss["columna"].astype(str)[::-1], df_miss["pct_nulos"][::-1])
    plt.xlabel("% nulos (muestra)")
    ensure_fig_saved(filename, title)

plot_missing(miss_lic, "fig_missing_top25_lic.png", "Fig. {A}.2 — LIC: Top 25 columnas con mayor % nulos (muestra)")
plot_missing(miss_oc,  "fig_missing_top25_oc.png",  "Fig. {A}.3 — OC: Top 25 columnas con mayor % nulos (muestra)")


## 4 — Distribución de montos (percentiles + boxplot + histogramas)

**Regla**: siempre segmentar por `moneda_norm`. Prohibido mezclar monedas.


In [11]:
# =========================================
# 4.1 - Detectar columna de monto y moneda
# =========================================
def pick_first(cols, cands):
    s = {c.lower(): c for c in cols}
    for x in cands:
        if x.lower() in s:
            return s[x.lower()]
    return None

lic_mon = pick_first(lic_cols, ["moneda_norm", "moneda"])
oc_mon  = pick_first(oc_cols,  ["moneda_norm", "moneda"])

lic_amount = pick_first(lic_cols, ["monto_total", "monto", "monto_licitacion", "monto_adjudicado", "monto_neto"])
oc_amount  = pick_first(oc_cols,  ["monto_total", "monto", "monto_oc", "monto_orden_compra", "monto_neto"])

assert lic_mon and oc_mon, "NO_CONSTA: falta moneda_norm/moneda en alguna vista"
assert lic_amount and oc_amount, "NO_CONSTA: falta columna de monto en alguna vista"

lic_mon, lic_amount, oc_mon, oc_amount


('moneda_norm', 'monto_adjudicado', 'moneda_norm', 'monto_total')

In [12]:
# =========================================
# 4.2 - Percentiles por moneda (LIC/OC)
# =========================================
def percentiles_por_moneda(view: str, moneda_col: str, amount_col: str):
    q = f"""
    SELECT
      {moneda_col}::text AS moneda_norm,
      COUNT(*)::bigint AS n,
      AVG({amount_col})::numeric AS avg_monto,
      STDDEV({amount_col})::numeric AS std_monto,
      PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY {amount_col}) AS p50,
      PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY {amount_col}) AS p90,
      PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY {amount_col}) AS p95,
      PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY {amount_col}) AS p99,
      MAX({amount_col}) AS max_monto
    FROM dw_sem.{view}
    WHERE {amount_col} IS NOT NULL
    GROUP BY 1
    ORDER BY n DESC;
    """
    return read_sql_df(q)

pct_lic = percentiles_por_moneda("v_fact_licitaciones_sem_v2", lic_mon, lic_amount)
pct_oc  = percentiles_por_moneda("v_fact_ordenes_compra_sem_v2", oc_mon,  oc_amount)

safe_to_csv(pct_lic, "tabla_percentiles_montos_lic.csv")
safe_to_csv(pct_oc,  "tabla_percentiles_montos_oc.csv")

pct_lic, pct_oc


(  moneda_norm        n     avg_monto     std_monto         p50          p90          p95           p99     max_monto
 0         CLP  4025579  1.165082e+08  4.229900e+09  14824438.0  214291450.0  326539040.0  1.606701e+09  3.357381e+12
 1         USD     6222  2.626140e+06  1.038127e+07     42449.0    3725100.0   21965068.0  3.475522e+07  2.099572e+08
 2         CLF     3545  3.156597e+07  1.037721e+09      2000.0      20439.0      51582.0  4.470588e+07  3.568134e+10
 3        None      441  3.731126e+06  4.093812e+07      4345.0     232566.0     809276.0  4.578324e+06  4.956160e+08,
   moneda_norm        n     avg_monto     std_monto         p50          p90           p95           p99     max_monto
 0         CLP  4508018  6.133929e+06  3.116620e+08  1218465.99  8886013.693  1.886388e+07  6.735281e+07  4.035709e+11
 1         USD    12744  7.503523e+04  7.542699e+05     6240.00    77637.686  2.009160e+05  1.281130e+06  3.732732e+07
 2         CLF    11901  4.815309e+03  1.268484e+05 

In [13]:
# =========================================
# 4.3 - Boxplot por moneda_norm (LIC/OC) (muestra)
# =========================================
def boxplot_por_moneda(df: pd.DataFrame, moneda_col: str, amount_col: str, filename: str, title: str, max_points=20000):
    if len(df) > max_points:
        df = df.sample(max_points, random_state=SAMPLE_SEED)
    df = df[[moneda_col, amount_col]].dropna()
    groups, labels = [], []
    for m, sub in df.groupby(moneda_col):
        vals = pd.to_numeric(sub[amount_col], errors="coerce").dropna().values
        if len(vals) > 0:
            groups.append(vals)
            labels.append(str(m))
    plt.figure(figsize=(10,4))
    plt.boxplot(groups, labels=labels, showfliers=True)
    plt.ylabel(amount_col)
    ensure_fig_saved(filename, title)

boxplot_por_moneda(df_lic_s, lic_mon, lic_amount, "fig_boxplot_montos_lic.png", "Fig. {A}.4 — LIC: Boxplot montos por moneda_norm (muestra)")
boxplot_por_moneda(df_oc_s,  oc_mon,  oc_amount,  "fig_boxplot_montos_oc.png",  "Fig. {A}.5 — OC: Boxplot montos por moneda_norm (muestra)")


In [14]:
# =========================================
# 4.4 - Histogramas por moneda_norm (LIC/OC) (muestra) — un PNG por moneda
# =========================================
def hist_por_moneda(df: pd.DataFrame, moneda_col: str, amount_col: str, filename_prefix: str, title_prefix: str, bins=40, max_points=30000):
    if len(df) > max_points:
        df = df.sample(max_points, random_state=SAMPLE_SEED)
    df = df[[moneda_col, amount_col]].dropna()
    for m, sub in df.groupby(moneda_col):
        vals = pd.to_numeric(sub[amount_col], errors="coerce").dropna().values
        if len(vals) == 0:
            continue
        plt.figure(figsize=(8,4))
        plt.hist(vals, bins=bins)
        plt.xlabel(amount_col)
        plt.ylabel("Frecuencia")
        ensure_fig_saved(f"{filename_prefix}_{m}.png", f"{title_prefix} — moneda_norm={m} (muestra)")

hist_por_moneda(df_lic_s, lic_mon, lic_amount, "fig_hist_montos_lic", "Fig. {A}.6 — LIC: Histograma de montos", bins=40)
hist_por_moneda(df_oc_s,  oc_mon,  oc_amount,  "fig_hist_montos_oc",  "Fig. {A}.7 — OC: Histograma de montos",  bins=40)


## 5 — Outliers (Top 20 montos por moneda) con contexto

**Objetivo**: transparencia de valores extremos (sin tratamiento). Se incluyen campos contextuales.


In [15]:
# =========================================
# 5.1 - Detectar columnas de fecha SK (para JOIN dim_fecha)
# =========================================
lic_fecha_sk = pick_first(lic_cols, ["fecha_publicacion_sk", "fecha_creacion_sk", "fecha_sk"])
oc_fecha_sk  = pick_first(oc_cols,  ["fecha_creacion_sk", "fecha_emision_sk", "fecha_sk"])

assert lic_fecha_sk and oc_fecha_sk, "NO_CONSTA: falta fecha_*_sk en alguna fact SEM"

# Detectar dim_fecha
df_fecha_cols = read_sql_df("""
SELECT column_name
FROM information_schema.columns
WHERE table_schema='dw' AND table_name='dim_fecha'
ORDER BY ordinal_position;
""")
fecha_cols = df_fecha_cols["column_name"].tolist()
fecha_key  = pick_first(fecha_cols, ["fecha_sk"])
fecha_date = pick_first(fecha_cols, ["fecha"])

if not (fecha_key and fecha_date):
    print("NO_CONSTA: dw.dim_fecha no expone fecha_sk/fecha. Se exportará outlier sin año_mes.")

lic_fecha_sk, oc_fecha_sk, fecha_key, fecha_date


('fecha_publicacion_sk', 'fecha_creacion_sk', 'fecha_sk', 'fecha')

In [16]:
# =========================================
# 5.2 - Top 20 outliers (LIC/OC) — se exporta un CSV por dominio
# =========================================
def top20_outliers(view: str, moneda_col: str, amount_col: str, fecha_sk_col: str):
    q = f"""
    WITH base AS (
      SELECT
        f.{moneda_col}::text AS moneda_norm,
        f.{amount_col}::numeric AS monto,
        f.organismo_sk,
        f.proveedor_sk,
        f.{fecha_sk_col} AS fecha_sk
      FROM dw_sem.{view} f
      WHERE f.{amount_col} IS NOT NULL
    )
    SELECT
      b.*,
      d.nombre_organismo,
      CASE WHEN {("TRUE" if (fecha_key and fecha_date) else "FALSE")}
           THEN to_char(df.{fecha_date}, 'YYYY-MM')
           ELSE NULL::text
      END AS anio_mes
    FROM base b
    LEFT JOIN dw_sem.v_dim_organismo_sem_v2 d ON b.organismo_sk = d.organismo_sk
    LEFT JOIN dw.dim_fecha df ON b.fecha_sk = df.{fecha_key}
    ORDER BY monto DESC
    LIMIT 20;
    """
    return read_sql_df(q)

df_out_lic = top20_outliers("v_fact_licitaciones_sem_v2", lic_mon, lic_amount, lic_fecha_sk)
df_out_oc  = top20_outliers("v_fact_ordenes_compra_sem_v2", oc_mon,  oc_amount,  oc_fecha_sk)

safe_to_csv(df_out_lic, "tabla_outliers_lic_top20.csv")
safe_to_csv(df_out_oc,  "tabla_outliers_oc_top20.csv")

df_out_lic, df_out_oc


(   moneda_norm         monto  organismo_sk  proveedor_sk  fecha_sk                                  nombre_organismo anio_mes
 0          CLP  3.357381e+12           156           882       358  SERVICIO LOCAL DE EDUCACION PUBLICA DE COLCHAGUA  2025-07
 1          CLP  3.357381e+12           156          2164       358  SERVICIO LOCAL DE EDUCACION PUBLICA DE COLCHAGUA  2025-07
 2          CLP  1.102918e+12           121         18331        84          JUNTA NACIONAL DE AUXILIO ESCOLAR Y BECA  2024-11
 3          CLP  1.102918e+12           121         18331        84          JUNTA NACIONAL DE AUXILIO ESCOLAR Y BECA  2024-11
 4          CLP  1.102918e+12           121         49226        84          JUNTA NACIONAL DE AUXILIO ESCOLAR Y BECA  2024-11
 5          CLP  1.102918e+12           121         49226        84          JUNTA NACIONAL DE AUXILIO ESCOLAR Y BECA  2024-11
 6          CLP  1.102918e+12           121          7242        84          JUNTA NACIONAL DE AUXILIO ESCOLAR 

## 6 — Concentración (Top 10 organismos y curva acumulada)

**Objetivo**: evidenciar concentración institucional (% del monto) por `moneda_norm`.


In [17]:
# =========================================
# 6.1 - Top 10 organismos por monto (por moneda)
# =========================================
def concentracion_top_org(view: str, moneda_col: str, amount_col: str, topk: int = 10):
    q = f"""
    WITH agg AS (
      SELECT
        f.{moneda_col}::text AS moneda_norm,
        d.nombre_organismo,
        SUM(f.{amount_col})::numeric AS total_monto
      FROM dw_sem.{view} f
      JOIN dw_sem.v_dim_organismo_sem_v2 d ON f.organismo_sk = d.organismo_sk
      WHERE f.{amount_col} IS NOT NULL
      GROUP BY 1,2
    ),
    ranked AS (
      SELECT *,
             ROW_NUMBER() OVER (PARTITION BY moneda_norm ORDER BY total_monto DESC) AS rk,
             SUM(total_monto) OVER (PARTITION BY moneda_norm) AS total_moneda
      FROM agg
    )
    SELECT
      moneda_norm,
      nombre_organismo,
      total_monto,
      (total_monto / NULLIF(total_moneda,0)) AS pct_sobre_moneda,
      rk
    FROM ranked
    WHERE rk <= {topk}
    ORDER BY moneda_norm, rk;
    """
    return read_sql_df(q)

conc_lic = concentracion_top_org("v_fact_licitaciones_sem_v2", lic_mon, lic_amount, 10)
conc_oc  = concentracion_top_org("v_fact_ordenes_compra_sem_v2", oc_mon,  oc_amount,  10)

safe_to_csv(conc_lic, "tabla_concentracion_top10_org_lic.csv")
safe_to_csv(conc_oc,  "tabla_concentracion_top10_org_oc.csv")

conc_lic.head(12), conc_oc.head(12)


(   moneda_norm                              nombre_organismo   total_monto  pct_sobre_moneda  rk
 0          CLF                      I MUNICIPALIDAD DE MAIPU  1.070440e+11          0.956593   1
 1          CLF                        CORP NACIONAL FORESTAL  1.356000e+09          0.012118   2
 2          CLF                     I MUNICIPALIDAD DE TEMUCO  1.002572e+09          0.008959   3
 3          CLF                     I MUNICIPALIDAD DE OSORNO  5.905588e+08          0.005277   4
 4          CLF                 I MUNICIPALIDAD DE CASABLANCA  3.629033e+08          0.003243   5
 5          CLF       SERVICIO DE SALUD VINA DEL MAR QUILLOTA  2.706453e+08          0.002419   6
 6          CLF  EMPRESA DE TRANSPORTE DE PASAJEROS METRO S A  2.316998e+08          0.002071   7
 7          CLF                  I MUNICIPALIDAD DE PENALOLEN  2.278701e+08          0.002036   8
 8          CLF                  SUBSECRETARIA DE TRANSPORTES  2.100287e+08          0.001877   9
 9          CLF     

In [18]:
# =========================================
# 6.2 - Curva acumulada (% acumulado Top 10) — un PNG por moneda
# =========================================
def curva_concentracion(df: pd.DataFrame, filename_prefix: str, title_prefix: str):
    for m, sub in df.groupby("moneda_norm"):
        sub = sub.sort_values("rk").copy()
        sub["pct_acum"] = sub["pct_sobre_moneda"].cumsum() * 100
        plt.figure(figsize=(8,4))
        plt.plot(sub["rk"], sub["pct_acum"])
        plt.xticks(sub["rk"])
        plt.ylim(0, 105)
        plt.xlabel("Ranking organismo (Top 10)")
        plt.ylabel("% acumulado del monto (sobre la moneda)")
        ensure_fig_saved(f"{filename_prefix}_{m}.png", f"{title_prefix} — moneda_norm={m}")

curva_concentracion(conc_lic, "fig_concentracion_acum_lic", "Fig. {A}.8 — LIC: Concentración acumulada Top 10 organismos")
curva_concentracion(conc_oc,  "fig_concentracion_acum_oc",  "Fig. {A}.9 — OC: Concentración acumulada Top 10 organismos")


## 7 — Dinámica temporal (operaciones y montos por mes)

**Objetivo**: describir evolución mensual del universo SEM.  
**Regla**: montos siempre por `moneda_norm` (sin mezcla).


In [19]:
# =========================================
# 7.1 - Serie mensual de operaciones (LIC vs OC)
# =========================================
assert fecha_key and fecha_date, "NO_CONSTA: dim_fecha sin fecha_sk/fecha; no se puede construir serie mensual"

q_ops = f"""
WITH lic AS (
  SELECT to_char(d.{fecha_date}, 'YYYY-MM') AS periodo,
         COUNT(*)::bigint AS n_lic
  FROM dw_sem.v_fact_licitaciones_sem_v2 f
  JOIN dw.dim_fecha d ON f.{lic_fecha_sk} = d.{fecha_key}
  GROUP BY 1
),
oc AS (
  SELECT to_char(d.{fecha_date}, 'YYYY-MM') AS periodo,
         COUNT(*)::bigint AS n_oc
  FROM dw_sem.v_fact_ordenes_compra_sem_v2 f
  JOIN dw.dim_fecha d ON f.{oc_fecha_sk} = d.{fecha_key}
  GROUP BY 1
)
SELECT COALESCE(lic.periodo, oc.periodo) AS periodo,
       COALESCE(n_lic,0) AS n_lic,
       COALESCE(n_oc,0)  AS n_oc
FROM lic
FULL OUTER JOIN oc USING (periodo)
ORDER BY 1;
"""
df_ops = read_sql_df(q_ops)
safe_to_csv(df_ops, "tabla_dinamica_operaciones_por_mes.csv")

plt.figure(figsize=(10,4))
plt.plot(df_ops["periodo"].astype(str), df_ops["n_lic"], label="LIC")
plt.plot(df_ops["periodo"].astype(str), df_ops["n_oc"],  label="OC")
plt.xticks(rotation=60, ha="right")
plt.ylabel("Nº operaciones")
plt.legend()
ensure_fig_saved("fig_dinamica_operaciones_por_mes.png", "Fig. {A}.10 — Dinámica mensual: nº operaciones (LIC vs OC)")


PosixPath('/home/engineer/Documents/Proyectos/TFM/docs/evidence/Fase6_EDA_CLASICO/outputs/ANEXO_EDA_SEM/fig_dinamica_operaciones_por_mes.png')

In [20]:
# =========================================
# 7.2 - Serie mensual de montos por moneda (LIC y OC separados) — un PNG por moneda
# =========================================
def montos_por_mes_y_moneda(view: str, moneda_col: str, amount_col: str, fecha_sk_col: str):
    q = f"""
    SELECT
      to_char(d.{fecha_date}, 'YYYY-MM') AS periodo,
      f.{moneda_col}::text AS moneda_norm,
      SUM(f.{amount_col})::numeric AS total_monto
    FROM dw_sem.{view} f
    JOIN dw.dim_fecha d ON f.{fecha_sk_col} = d.{fecha_key}
    WHERE f.{amount_col} IS NOT NULL
    GROUP BY 1,2
    ORDER BY 1,2;
    """
    return read_sql_df(q)

df_m_lic = montos_por_mes_y_moneda("v_fact_licitaciones_sem_v2", lic_mon, lic_amount, lic_fecha_sk)
df_m_oc  = montos_por_mes_y_moneda("v_fact_ordenes_compra_sem_v2", oc_mon,  oc_amount,  oc_fecha_sk)

safe_to_csv(df_m_lic, "tabla_dinamica_montos_lic_por_mes_moneda.csv")
safe_to_csv(df_m_oc,  "tabla_dinamica_montos_oc_por_mes_moneda.csv")

def plot_montos(df: pd.DataFrame, filename_prefix: str, title_prefix: str):
    for m, sub in df.groupby("moneda_norm"):
        plt.figure(figsize=(10,4))
        plt.plot(sub["periodo"].astype(str), pd.to_numeric(sub["total_monto"], errors="coerce"))
        plt.xticks(rotation=60, ha="right")
        plt.ylabel("Monto total")
        ensure_fig_saved(f"{filename_prefix}_{m}.png", f"{title_prefix} — moneda_norm={m}")

plot_montos(df_m_lic, "fig_dinamica_montos_lic", "Fig. {A}.11 — LIC: Montos por mes")
plot_montos(df_m_oc,  "fig_dinamica_montos_oc",  "Fig. {A}.12 — OC: Montos por mes")
